In [5]:
import sys
import os
import lm_eval
from lm_eval.models.huggingface import HFLM

# Add current directory to path so we can import local files
sys.path.append(os.getcwd())

# Import your function from llama_1b.py
from llama_1b import load_llama_1b
from llama_8b import load_llama_8b
print("Imports complete.")


Imports complete.


In [9]:
# Load the model using your custom function
model, tokenizer = load_llama_1b()

print("✅ Model loaded into VRAM.")

Loading Llama-3.2-1B-Instruct in FP16 (Student)...
✅ Model loaded into VRAM.


In [10]:
# Wrap the pre-loaded model for lm-eval
# we pass the object directly to 'pretrained'
lm_eval_model = HFLM(
    pretrained=model,
    tokenizer=tokenizer,
    batch_size="auto",
    device="cuda"       # Let it figure out max batch size
)

print("✅ Model wrapped for MMLU.")

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


✅ Model wrapped for MMLU.


In [11]:
print("Starting MMLU Benchmark...")

# Select tasks. 
# Use 'mmlu' for the full 57-task benchmark.
# Use 'mmlu_abstract_algebra' etc. for specific subjects.
task_selection = ["mmlu"] 

results = lm_eval.simple_evaluate(
    model=lm_eval_model,
    tasks=task_selection,
    num_fewshot=5,      # Standard MMLU uses 5-shot
    limit=5             # <--- IMPORTANT: REMOVE THIS LINE FOR FULL RUN
)

print("✅ Benchmark Complete.")

Starting MMLU Benchmark...


Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
Overwriting default num_fewshot of mmlu_anatomy from None to 5
Overwriting default num_fewshot of mmlu_astronomy from None to 5
Overwriting default num_fewshot of mmlu_college_biology from None to 5
Overwriting default num_fewshot of mmlu_college_chemistry from None to 5
Overwriting default num_fewshot of mmlu_college_computer_science from None to 5
Overwriting default num_fewshot of mmlu_college_mathematics from None to 5
Overwriting default num_fewshot of mmlu_college_physics from None to 5
Overwriting default num_fewshot of mmlu_computer_security from None to 5
Overwriting default num_fewshot of mmlu_conceptual_physics from None to 5
Overwriting default num_fewshot of mmlu_electrical_engineering from None to 5
Overwriting default num_fewshot of mmlu_elementary_mathematics from None to 5
Overwriting default num_fewshot of mmlu_high_school_biology from None to 5
Overwriting default num_fewshot of mmlu_high_school_

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 8


Running loglikelihood requests: 100%|██████████| 1140/1140 [20:37<00:00,  1.09s/it]


✅ Benchmark Complete.


In [13]:
import pandas as pd
import json

# 1. Print a pretty summary table
print(lm_eval.utils.make_table(results))

# 2. Extract the final accuracy score
if 'groups' in results and 'mmlu' in results['groups']:
    final_acc = results['groups']['mmlu']['acc,none']
    print(f"\n🏆 Final MMLU Average Accuracy: {final_acc:.2%}")

# 3. Save detailed JSON to disk (FIXED)
with open("mmlu_results_llama1b.json", "w") as f:
    # We add default=str to handle the torch.dtype error
    json.dump(results, f, indent=2, default=str) 

print("\nResults saved to mmlu_results_llama1b.json")

|                 Tasks                  |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|----------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                    |      2|none  |      |acc   |↑  |0.4281|±  |0.0289|
|mmlu_humanities                         |      2|none  |      |acc   |↑  |0.4462|±  |0.0606|
|mmlu_formal_logic                       |      1|none  |     5|acc   |↑  |0.2000|±  |0.2000|
|mmlu_high_school_european_history       |      1|none  |     5|acc   |↑  |0.8000|±  |0.2000|
|mmlu_high_school_us_history             |      1|none  |     5|acc   |↑  |0.6000|±  |0.2449|
|mmlu_high_school_world_history          |      1|none  |     5|acc   |↑  |0.8000|±  |0.2000|
|mmlu_international_law                  |      1|none  |     5|acc   |↑  |0.6000|±  |0.2449|
|mmlu_jurisprudence                      |      1|none  |     5|acc   |↑  |0.4000|±  |0.2449|
|mmlu_logical_fallacies                  |      1|none  |   

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model():
    model_path = "./distilled_llama_proper"
    
    distil_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    distil_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="cuda",
        torch_dtype=torch.bfloat16
    )

    return distil_model, distil_tokenizer

if __name__ == "__main__":
    load_model()

g:\T2430392\Efficient-Distilled-miniLLM-to-Prevent-Jailbreak-Attacks\jailbreak\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer you are loading from './distilled_llama_proper' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
import sys
import os
import json
import lm_eval
from lm_eval.models.huggingface import HFLM

sys.path.append(os.getcwd())


distil_model, distil_tokenizer = load_model()

lm_eval_model = HFLM(
    pretrained=distil_model,
    tokenizer=distil_tokenizer,
    batch_size="auto",
    device="cuda"
)

results = lm_eval.simple_evaluate(
    model=lm_eval_model,
    tasks=["mmlu"],
    num_fewshot=5
)

print(lm_eval.utils.make_table(results))

if 'groups' in results and 'mmlu' in results['groups']:
    print(results['groups']['mmlu']['acc,none'])

with open("mmlu_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

The tokenizer you are loading from './distilled_llama_proper' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
Overwriting default num_fewshot of mmlu_anatomy from None to 5
Overwriting default num_fewshot of mmlu_astronomy from None to 5
Overwriting default num_fewshot of mmlu_college_biology from None to 5
Overwriting default num_fewshot of mmlu_college_chemistry fr

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 9


Running loglikelihood requests: 100%|██████████| 56168/56168 [3:31:14<00:00,  4.43it/s]  


|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.4353|±  |0.0041|
| - humanities                          |      2|none  |      |acc   |↑  |0.3964|±  |0.0070|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.2619|±  |0.0393|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.4909|±  |0.0390|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |0.5539|±  |0.0349|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |0.4937|±  |0.0325|
|  - international_law                  |      1|none  |     5|acc   |↑  |0.6116|±  |0.0445|
|  - jurisprudence                      |      1|none  |     5|acc   |↑  |0.5093|±  |0.0483|
|  - logical_fallacies                  |      1|none  |     5|acc   |

In [1]:

import sys
import os
import lm_eval
from lm_eval.models.huggingface import HFLM

# Add current directory to path so we can import local files
sys.path.append(os.getcwd())

#load teacher model
from llama_8b import load_llama_8b

print("Imports complete.")

# Load the model using your custom function
model, tokenizer = load_llama_8b()

print("✅ Model loaded into VRAM.")

Imports complete.
Loading Meta-Llama-3.1-8B-Instruct in FP16 (Teacher)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:32<00:00,  8.15s/it]


✅ Model loaded into VRAM.


In [2]:
# Wrap the pre-loaded model for lm-eval
# we pass the object directly to 'pretrained'
lm_eval_model = HFLM(
    pretrained=model,
    tokenizer=tokenizer,
    batch_size="auto",
    device="cuda"       # Let it figure out max batch size
)

print("✅ Model wrapped for MMLU.")

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


✅ Model wrapped for MMLU.


In [3]:
print("Starting MMLU Benchmark...")

# Select tasks. 
# Use 'mmlu' for the full 57-task benchmark.
# Use 'mmlu_abstract_algebra' etc. for specific subjects.
task_selection = ["mmlu"] 

results = lm_eval.simple_evaluate(
    model=lm_eval_model,
    tasks=task_selection,
    num_fewshot=5,      # Standard MMLU uses 5-shot
    limit=5             # <--- IMPORTANT: REMOVE THIS LINE FOR FULL RUN
)

print("✅ Benchmark Complete.")

Starting MMLU Benchmark...


Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
Overwriting default num_fewshot of mmlu_anatomy from None to 5
Overwriting default num_fewshot of mmlu_astronomy from None to 5
Overwriting default num_fewshot of mmlu_college_biology from None to 5
Overwriting default num_fewshot of mmlu_college_chemistry from None to 5
Overwriting default num_fewshot of mmlu_college_computer_science from None to 5
Overwriting default num_fewshot of mmlu_college_mathematics from None to 5
Overwriting default num_fewshot of mmlu_college_physics from None to 5
Overwriting default num_fewshot of mmlu_computer_security from None to 5
Overwriting default num_fewshot of mmlu_conceptual_physics from None to 5
Overwriting default num_fewshot of mmlu_electrical_engineering from None to 5
Overwriting default num_fewshot of mmlu_elementary_mathematics from None to 5
Overwriting default num_fewshot of mmlu_high_school_biology from None to 5
Overwriting default num_fewshot of mmlu_high_school_

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 6


Running loglikelihood requests: 100%|██████████| 1140/1140 [36:06<00:00,  1.90s/it]   


✅ Benchmark Complete.


In [4]:
import pandas as pd
import json

# 1. Print a pretty summary table
print(lm_eval.utils.make_table(results))

# 2. Extract the final accuracy score
if 'groups' in results and 'mmlu' in results['groups']:
    final_acc = results['groups']['mmlu']['acc,none']
    print(f"\n🏆 Final MMLU Average Accuracy: {final_acc:.2%}")

# 3. Save detailed JSON to disk (FIXED)
with open("mmlu_results_llama8b.json", "w") as f:
    # We add default=str to handle the torch.dtype error
    json.dump(results, f, indent=2, default=str) 

print("\nResults saved to mmlu_results_llama8b.json")

|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.7158|±  |0.0264|
| - humanities                          |      2|none  |      |acc   |↑  |0.7692|±  |0.0487|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.4000|±  |0.2449|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.8000|±  |0.2000|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |1.0000|±  |0.0000|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |1.0000|±  |0.0000|
|  - international_law                  |      1|none  |     5|acc   |↑  |1.0000|±  |0.0000|
|  - jurisprudence                      |      1|none  |     5|acc   |↑  |0.4000|±  |0.2449|
|  - logical_fallacies                  |      1|none  |     5|acc   |